In [1]:
def check_pdbqt_format(file_path):
    with open(file_path) as f:
        lines = f.readlines()

    atom_lines = [l for l in lines if l.startswith(('ATOM', 'HETATM'))]
    problems = []
    for i, line in enumerate(atom_lines, start=1):
        parts = line.split()
        if len(parts) < 11:
            problems.append((i, line.strip()))

    print(f"📘 {file_path}")
    print(f"Total ATOM/HETATM lines: {len(atom_lines)}")
    if problems:
        print("⚠️ Lines with missing fields (possibly incorrect PDBQT formatting):")
        for i, l in problems[:10]:
            print(f"Line {i}: {l}")
    else:
        print("✅ All atom lines look correctly formatted.\n")

check_pdbqt_format("2FOMiteration1.pdbqt")
check_pdbqt_format("remdesiviriteration1.pdbqt")


📘 2FOMiteration1.pdbqt
Total ATOM/HETATM lines: 1718
✅ All atom lines look correctly formatted.

📘 remdesiviriteration1.pdbqt
Total ATOM/HETATM lines: 47
✅ All atom lines look correctly formatted.



In [2]:
def check_ligand_structure(file_path):
    with open(file_path) as f:
        txt = f.read()

    print(f"📗 {file_path}")
    print("Contains ROOT:", "ROOT" in txt)
    print("Contains ENDROOT:", "ENDROOT" in txt)
    print("Contains BRANCH:", "BRANCH" in txt)
    print("Contains ENDBRANCH:", "ENDBRANCH" in txt)
    print("Contains charges (e.g. .00, .50, -.):", any(c in txt for c in ['.00', '.50', '-.']))
    print("-" * 50)

check_ligand_structure("remdesiviriteration1.pdbqt")


📗 remdesiviriteration1.pdbqt
Contains ROOT: True
Contains ENDROOT: True
Contains BRANCH: True
Contains ENDBRANCH: True
Contains charges (e.g. .00, .50, -.): True
--------------------------------------------------


In [3]:
import numpy as np

ligand_file = "remdesiviriteration1.pdbqt"
receptor_file = "2FOMiteration1.pdbqt"

coords = []
with open(ligand_file) as f:
    for line in f:
        if line.startswith(("ATOM", "HETATM")):
            try:
                x = float(line[30:38].strip())
                y = float(line[38:46].strip())
                z = float(line[46:54].strip())
                coords.append((x, y, z))
            except:
                parts = line.split()
                if len(parts) >= 7:
                    coords.append((float(parts[-6]), float(parts[-5]), float(parts[-4])))

coords = np.array(coords)
centroid = coords.mean(axis=0)
mins, maxs = coords.min(axis=0), coords.max(axis=0)
ligand_size = maxs - mins
margin = 8.0  # Ångström padding around ligand

box_size = ligand_size + margin
center_x, center_y, center_z = centroid.tolist()
size_x, size_y, size_z = box_size.tolist()

print("📍 Center (x,y,z):", center_x, center_y, center_z)
print("📦 Box size (x,y,z):", size_x, size_y, size_z)


📍 Center (x,y,z): -0.29772340425531896 0.006212765957446677 0.07119148936170211
📦 Box size (x,y,z): 24.616 18.463 13.355


In [4]:
# Option A: compute centroid from co-crystallized ligand in original PDB
import numpy as np, os, re

pdb_original = "2FOM.pdb"   # change to your original PDB filename
ligand_resnames_to_ignore = {"HOH", "W", "NA", "CL", "MG", "CA"}  # ignore waters/ions

if not os.path.exists(pdb_original):
    print("Original PDB not found:", pdb_original)
else:
    coords_by_res = {}
    with open(pdb_original) as f:
        for line in f:
            if line.startswith("HETATM"):
                resname = line[17:20].strip()
                resnum = line[22:26].strip()
                if resname in ligand_resnames_to_ignore:
                    continue
                try:
                    x = float(line[30:38]); y = float(line[38:46]); z = float(line[46:54])
                except:
                    continue
                key = (resname, resnum)
                coords_by_res.setdefault(key, []).append((x,y,z))

    # pick the largest non-water hetero-residue cluster as ligand
    candidate = None
    max_atoms = 0
    for key, coords in coords_by_res.items():
        if len(coords) > max_atoms:
            max_atoms = len(coords); candidate = key

    if candidate is None:
        print("No heteroatoms (non-water) found in", pdb_original)
    else:
        coords = np.array(coords_by_res[candidate])
        centroid = coords.mean(axis=0)
        mins = coords.min(axis=0); maxs = coords.max(axis=0)
        ligand_size = maxs - mins
        margin = 8.0
        box_size = ligand_size + margin
        print("Using ligand:", candidate)
        print("Centroid (x,y,z):", centroid.tolist())
        print("Ligand size (xyz):", ligand_size.tolist())
        print("Suggested box size (xyz):", box_size.tolist())


Using ligand: ('GOL', '202')
Centroid (x,y,z): [-3.7230000000000003, -9.954833333333333, 2.2398333333333333]
Ligand size (xyz): [2.9269999999999996, 3.3420000000000005, 2.1899999999999995]
Suggested box size (xyz): [10.927, 11.342, 10.19]


In [5]:
# Paste & run to inspect heteroatoms and their counts in the original PDB
pdb_original = "2FOM.pdb"   # change if needed
from collections import Counter, defaultdict
res_counts = Counter()
res_examples = defaultdict(list)

with open(pdb_original) as f:
    for line in f:
        if line.startswith("HETATM"):
            resname = line[17:20].strip()
            resnum = line[22:26].strip()
            key = (resname, resnum)
            res_counts[key] += 1
            if len(res_examples[key]) < 5:
                res_examples[key].append(line.strip())

print("HETATM residue counts (resname, resnum) -> count\n")
for k, c in res_counts.most_common():
    print(f"{k} -> {c}")
print("\nExample lines for GOL 202 (if present):")
for l in res_examples.get(("GOL","202"), []):
    print(l)


HETATM residue counts (resname, resnum) -> count

('GOL', '202') -> 6
('GOL', '203') -> 6
('HOH', '147') -> 2
('HOH', '148') -> 2
('HOH', '307') -> 2
('HOH', '308') -> 2
('CL', '201') -> 1
('HOH', '105') -> 1
('HOH', '106') -> 1
('HOH', '107') -> 1
('HOH', '108') -> 1
('HOH', '109') -> 1
('HOH', '110') -> 1
('HOH', '111') -> 1
('HOH', '112') -> 1
('HOH', '113') -> 1
('HOH', '114') -> 1
('HOH', '115') -> 1
('HOH', '116') -> 1
('HOH', '117') -> 1
('HOH', '118') -> 1
('HOH', '119') -> 1
('HOH', '120') -> 1
('HOH', '121') -> 1
('HOH', '122') -> 1
('HOH', '123') -> 1
('HOH', '124') -> 1
('HOH', '125') -> 1
('HOH', '126') -> 1
('HOH', '127') -> 1
('HOH', '128') -> 1
('HOH', '129') -> 1
('HOH', '130') -> 1
('HOH', '131') -> 1
('HOH', '132') -> 1
('HOH', '133') -> 1
('HOH', '134') -> 1
('HOH', '135') -> 1
('HOH', '136') -> 1
('HOH', '137') -> 1
('HOH', '138') -> 1
('HOH', '139') -> 1
('HOH', '140') -> 1
('HOH', '141') -> 1
('HOH', '142') -> 1
('HOH', '143') -> 1
('HOH', '144') -> 1
('HOH', '14

In [6]:
receptor_file = "2FOMiteration1.pdbqt"
ligand_file   = "remdesiviriteration1.pdbqt"
config_file   = "configiteration1.txt"

center_x = -3.723
center_y = -9.954833333333333
center_z =  2.2398333333333333

# your suggested box (from earlier) — change margin if you want larger box
size_x = 10.927
size_y = 11.342
size_z = 10.190

# Optional: if you want a larger box, uncomment and change margin, e.g. margin = 10.0
# margin = 10.0

cfg = f"""receptor = {receptor_file}
ligand = {ligand_file}

center_x = {center_x:.3f}
center_y = {center_y:.3f}
center_z = {center_z:.3f}

size_x = {size_x:.3f}
size_y = {size_y:.3f}
size_z = {size_z:.3f}

exhaustiveness = 8
num_modes = 10
energy_range = 3
"""

with open(config_file, "w") as f:
    f.write(cfg)

print("Wrote", config_file)
print(cfg)


Wrote configiteration1.txt
receptor = 2FOMiteration1.pdbqt
ligand = remdesiviriteration1.pdbqt

center_x = -3.723
center_y = -9.955
center_z = 2.240

size_x = 10.927
size_y = 11.342
size_z = 10.190

exhaustiveness = 8
num_modes = 10
energy_range = 3



In [8]:
!"C:\Users\Ayush\vina.exe" --config configiteration1.txt --out outiteration3.pdbqt


AutoDock Vina v1.2.7
#################################################################
# If you used AutoDock Vina in your work, please cite:          #
#                                                               #
# J. Eberhardt, D. Santos-Martins, A. F. Tillack, and S. Forli  #
# AutoDock Vina 1.2.0: New Docking Methods, Expanded Force      #
# Field, and Python Bindings, J. Chem. Inf. Model. (2021)       #
# DOI 10.1021/acs.jcim.1c00203                                  #
#                                                               #
# O. Trott, A. J. Olson,                                        #
# AutoDock Vina: improving the speed and accuracy of docking    #
# with a new scoring function, efficient optimization and       #
# multithreading, J. Comp. Chem. (2010)                         #
# DOI 10.1002/jcc.21334                                         #
#                                                               #
# Please see https://github.com/ccsb-scripps/AutoDock-V